In [ ]:
import requests

GROQ_API_KEY = "SUA CHAVE DE API"

GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

MODEL = "llama-3.1-8b-instant"

def call_llm(prompt):
    response = requests.post(
        GROQ_URL,
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL,
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    if response.status_code != 200:
        raise Exception(f"Erro na API: {response.text}")

    return response.json()["choices"][0]["message"]["content"]

In [ ]:
def extract():
    print("\n Iniciando EXTRACT...\n")

    users = [
        {"id": 1, "name": "Ranni", "news": []},
        {"id": 2, "name": "Radahn", "news": []},
        {"id": 3, "name": "Malenia", "news": []},
        {"id": 4, "name": "Blaidd", "news": []},
        {"id": 5, "name": "Melina", "news": []}
    ]

    print(" Dados extraídos:")
    print("-" * 40)

    for user in users:
        print(f"ID: {user['id']} | Nome: {user['name']} | News: {user['news']}")

    print("\n EXTRACT finalizado.\n")

    return users

users = extract()

In [ ]:
import re


def build_prompt(user_name: str) -> str:
    return f"""
Gere uma frase curta e completa em português (máx 60 caracteres)
incentivando {user_name} a acumular runas.

Regras:
- Apenas uma frase
- Máx 60 caracteres
- Não usar aspas
- Deve terminar com ponto final
"""


def clean_response(text: str) -> str:
    text = text.strip().strip('"').strip("'")

    match = re.search(r"(.+?[.!?])", text)
    if match:
        text = match.group(1)

    if len(text) > 60:
        text = text[:60]
        text = text.rsplit(" ", 1)[0] + "."

    if not text.endswith("."):
        text += "."

    return text


def fallback_message(user_name: str) -> str:
    return f"{user_name}, reúna runas e fortaleça seu destino."


def transform(users: list) -> list:
    print("\nIniciando etapa de transformação...\n")

    for user in users:
        name = user["name"]
        print(f"Processando usuário: {name}")

        prompt = build_prompt(name)

        try:
            raw = call_llm(prompt)
            message = clean_response(raw)
            print(f"Mensagem gerada: {message}")

        except Exception as e:
            print(f"Erro ao gerar mensagem para {name}: {e}")
            message = fallback_message(name)
            print(f"Mensagem padrão aplicada: {message}")

        user["news"].append({
            "description": message
        })

    print("\nTransformação concluída.\n")

    return users

users = transform(users)

In [ ]:
import pandas as pd


def load(users: list) -> pd.DataFrame:
    print("\nIniciando etapa de carregamento...\n")

    rows = []

    for user in users:
        for news in user["news"]:
            rows.append({
                "id": user["id"],
                "name": user["name"],
                "message": news["description"]
            })

    df = pd.DataFrame(rows)

    output_file = "resultado_etl.csv"
    df.to_csv(output_file, index=False)

    print(f"Arquivo gerado: {output_file}")
    print("\nPré-visualização dos dados:\n")
    print(df.head())

    print("\nCarregamento concluído.\n")

    return df


df = load(users)